In [ ]:
%%capture
import os
from pathlib import Path

from dj_notebook import activate

env_file = os.environ["INTECOMM_ENV"]
analysis_folder = Path(os.environ["INTECOMM_ANALYSIS_FOLDER"])
reports_folder = Path(os.environ["INTECOMM_ANALYSIS_FOLDER"])
plus = activate(dotenv_file=env_file)

In [ ]:
from intecomm_analytics.dataframes import get_df_main_1858
from intecomm_analytics.stats_models import run_gee_identity_link_model, run_gee_guassian_model

In [ ]:
# run gee model unadjusted and adjusted (sex,age) and output
# the result plus counts (categorical/binary outcomes) and
# mean (continuous outcomes) to one PDF per outcome variable.

# the identity-binomial will output a DomainWarning that the
# Identity link function does not respect the domain of the
# Binomial family. This may safely be ignored.

In [ ]:
gee_results_path = analysis_folder / "gee_results"
if not gee_results_path.exists():
    raise ValueError("Path does not exist")

In [ ]:
df_main_orig = get_df_main_1858(None, fasting_hours=8.0)
df_base = df_main_orig.copy()

# note use of 'retained_12m' so numbers match 12m line in consort chart
# see get_df_main_1858 for how this column is generated
df_12m = df_main_orig[df_main_orig.retained_12m == 1].copy()
df_12m = df_12m.reset_index(drop=True)

In [ ]:
cohort_list = ['DM_ALONE', 'HTN_ALONE', 'HTN_DM']
assert len(df_base.query("primary_cohort_str.isin(@cohort_list)")) == 1211
assert len(df_12m.query("primary_cohort_str.isin(@cohort_list)")) == 1130

# composite baseline
run_gee_identity_link_model(df_base, "primary_composite_baseline", cohort_list, as_percentage=True, path=gee_results_path)

# composite endline
run_gee_identity_link_model(df_12m, "primary_composite_endline", cohort_list, as_percentage=True, path=gee_results_path)

In [ ]:
cohort_list = ['HTN_ALONE', 'HTN_DM']
assert len(df_base.query("primary_cohort_str.isin(@cohort_list)")) == 529 + 537
assert len(df_12m.query("primary_cohort_str.isin(@cohort_list)")) == 497 + 499

# baseline using df_base
for col in ["bp_controlled_baseline", "bp_severe_htn_baseline"]:
    run_gee_identity_link_model(df_base, col, cohort_list, as_percentage=True, path=gee_results_path)

for col in ["bp_sys_baseline", "bp_dia_baseline"]:
    run_gee_guassian_model(df_base, col, cohort_list, path=gee_results_path)

# endline using df_12m
for col in ["bp_controlled_endline", "bp_severe_htn_endline"]:
    run_gee_identity_link_model(df_12m, col, cohort_list, as_percentage=True, path=gee_results_path)

for col in ["bp_sys_endline", "bp_dia_endline"]:
    run_gee_guassian_model(df_12m, col, cohort_list, path=gee_results_path)


In [ ]:
cohort_list = ['DM_ALONE', 'HTN_DM']
assert len(df_base.query("primary_cohort_str.isin(@cohort_list)")) == 255 + 236
assert len(df_12m.query("primary_cohort_str.isin(@cohort_list)")) == 239 + 222

# baseline using df_base
for col in ["glucose_controlled_baseline"]:
    run_gee_identity_link_model(df_base, col, cohort_list, as_percentage=True, path=gee_results_path)

for col in ["glucose_value_baseline"]:
    run_gee_guassian_model(df_base, col, cohort_list, path=gee_results_path)

# endline using df_12m
for col in ["glucose_controlled_endline"]:
    run_gee_identity_link_model(df_12m, col, cohort_list, as_percentage=True, path=gee_results_path)

for col in ["glucose_value_endline"]:
    run_gee_guassian_model(df_12m, col, cohort_list, path=gee_results_path)



In [ ]:
cohort_list = ['HIV_ALONE']
assert len(df_base.query("primary_cohort_str.isin(@cohort_list)")) == 242 + 247
assert len(df_12m.query("primary_cohort_str.isin(@cohort_list)")) == 233 + 237

# baseline using df_base
for col in ["primary_vl_controlled_baseline", "vl_controlled_baseline_400", ]:
    run_gee_identity_link_model(df_base, col, cohort_list, as_percentage=True, path=gee_results_path)

# endline using df_12m
for col in ["primary_vl_controlled_endline", "vl_controlled_endline_400"]:
    run_gee_identity_link_model(df_12m, col, cohort_list, as_percentage=True, path=gee_results_path)

